<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 필요 패키지 설치 (오류 해결 핵심!)
!pip install pyreadstat

# 2. 구글 드라이브 마운트
from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat

drive.mount("/content/drive")

# 3. 데이터 불러오기 (본인 Google Drive 경로)
DATA_DIR = "/content/drive/MyDrive/hdac_data"
cols = [
    "YEAR",
    "STRATA",
    "CLUSTER",
    "W",
    "GRADE",
    "SEX",
    "AGE",
    "HT",
    "WT",
    "F_BR",
    "WC_MN",
]
df, meta = pyreadstat.read_sas7bdat(
    f"{DATA_DIR}/kyrbs2025.sas7bdat", usecols=cols
)

print("=== 1. 원자료 불러오기 성공 ===")
print(f"전체 참여자 수: {len(df):,}명")

# 4. 고등학생 도메인 지정 (GRADE: 4=고1, 5=고2, 6=고3)
df_hs = df[df["GRADE"].isin([4, 5, 6])].copy()
print(f"고등학생 도메인 대상자 수: {len(df_hs):,}명")

# 5. 아침식사 결식 재코딩 (F_BR)
# 원코드 1~5 -> 0 (비결식군: 주 0~4회), 6~8 -> 1 (결식군: 주 5~7회)
df_hs["breakfast_skip"] = np.where(
    df_hs["F_BR"].isin([6, 7, 8]),
    1,
    np.where(df_hs["F_BR"].isin([1, 2, 3, 4, 5]), 0, np.nan),
)

print("\n=== 2. 아침식사 결식 집단별 인원 ===")
print(
    df_hs["breakfast_skip"]
    .value_counts(dropna=False)
    .rename({0.0: "비결식군 (주 0~4회)", 1.0: "결식군 (주 5~7회)"})
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 13.3 MB/s  0:00:00
Mounted at /content/drive
=== 1. 원자료 불러오기 성공 ===
전체 참여자 수: 54,170명
고등학생 도메인 대상자 수: 25,843명

=== 2. 아침식사 결식 집단별 인원 ===
breakfast_skip
비결식군 (주 0~4회)    15296
결식군 (주 5~7회)     10547
Name: count, dtype: int64
